In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

Read and observe some basic properties of data

In [ ]:
df = pd.read_csv('./datasets/BostonHousing.csv')
df.shape

In [ ]:
df.info()

In [ ]:
target = df['medv']
X = df.drop(columns=['medv', 'nox', 'chas'],axis=0)

observe variance of every feature. variance near zero means it assign the same value for that specific feature to every sample.

In [ ]:
X.var().sort_values()

scaling feature values using standard sclaer(mean 0 and std 1)

In [ ]:
scaler = StandardScaler()
cols = [c for c in X.columns]

X[cols] = scaler.fit_transform(X[cols])

X['b'].describe()

1. check for each feature's variance. features with lower variance will tend to have a same value for every sample.

In [ ]:
X.var().sort_values()

2. check for feature correlation

In [ ]:
corr = X.corr().abs()
corr

3. check correlation of every feature with the target

In [ ]:
correlations = (
    X.join(target.rename('MEDV'))
    .corr()['MEDV']
    .drop('MEDV')
    .sort_values(key=abs, ascending=False)
)

print(correlations)

4. Permutation importance works by randomly shuffling the values of one feature and checking how much the model's performance gets worse.

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    model,
    X,
    target,
    scoring='neg_mean_squared_error',
    n_repeats=10,
    random_state=42
)

In [ ]:
from keras import models
from keras import layers


def build_model(X_train):

    model = models.Sequential()
    model.add(layers.Input(shape=(X_train.shape[1],)))
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dense(1))
    model.compile(optimizer='rmsprop', loss='mse', metrics=['mae'])

    return model

In [57]:
def k_fold_validation(train_data, train_targets, *, epochs=100 ,k=4, func=build_model):
    num_val_samples = len(train_data) // k
    all_scores = []

    for i in range(k):
        print(f'Processing fold #{i}')

        valid_x = train_data[i * num_val_samples : (i + 1) * num_val_samples]
        valid_y = train_targets[i * num_val_samples: (i+1) * num_val_samples]

        partial_x_train = np.concatenate((train_data.iloc[: i * num_val_samples], train_data[(i+1) * num_val_samples: ]))
        partial_y_train = np.concatenate((train_targets[: i * num_val_samples], train_targets[(i+1) * num_val_samples: ]))

        model = func(partial_x_train)
        model.fit(partial_x_train, partial_y_train,epochs=epochs, batch_size=1, verbose=0)

        val_mse, val_mae = model.evaluate(valid_x, valid_y, verbose=0)
        all_scores.append(val_mae)
    return all_scores


In [61]:
all_maes = k_fold_validation(X, target, epochs=5, k=4)

Processing fold #0
Processing fold #1
Processing fold #2
Processing fold #3


In [62]:
sum(all_maes) / len(all_maes)

3.3503312468528748